# 🚀 Fine-Tuning Qwen-4B with QLoRA (4-bit) for SaaS Presale AI Assistant

This notebook automates end-to-end LLM fine-tuning for **SaaS Presale Engineering**:
- **Base Model**: `Qwen/Qwen1.5-4B-Chat`
- **Quantization**: 4-bit NormalFloat (NF4) via `bitsandbytes`
- **PEFT Method**: QLoRA (Rank 16, Alpha 32)
- **Source Dataset**: `DeepMostInnovations/saas-sales-conversations`
- **Dataset Scope**: Exactly **2,000 conversations** processed into **10,000 instruction examples**:
  - **8,000 Train examples** (1,600 conversations)
  - **2,000 Validation examples** (400 conversations)
- **5 Presale Specialized Tasks**:
  1. 🎯 Pain Point Extraction
  2. 📋 Requirement Extraction
  3. 🔍 Intent Classification
  4. 🛡️ Objection & Risk Extraction
  5. 🏆 BANT Lead Qualification

---

In [ ]:
# Step 1: Install required libraries for QLoRA & SFTTrainer
!pip install -q --upgrade pip
!pip install -q torch transformers datasets peft bitsandbytes trl accelerate sentencepiece

import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"Device count: {torch.cuda.device_count()}")
!nvidia-smi


In [ ]:
# Step 2: Download & Preprocess 2,000 Conversations into 10,000 Examples (8k Train / 2k Val)
import json
import random
from typing import Dict, List, Any, Tuple
from datasets import load_dataset, Dataset

SYSTEM_PROMPTS = {
    "pain_point": "You are an expert AI Presale Analyst. Analyze the sales conversation transcript and identify the customer's key pain points, operational bottlenecks, and current frustrations.",
    "requirement": "You are an expert AI Presale Analyst. Analyze the sales conversation transcript and extract the functional and technical requirements/needs expressed by the customer.",
    "intent": "You are an expert AI Presale Analyst. Classify the customer's primary intent in this sales interaction into one of the following categories: [Product Capability Evaluation, Pricing & ROI Evaluation, Implementation & Onboarding Assessment, Initial Discovery, Commercial Negotiation]. Provide a concise justification based on the conversation.",
    "objection": "You are an expert AI Presale Analyst. Identify all objections, hesitations, risks, and concerns raised by the customer during this sales interaction regarding pricing, complexity, time, or change management.",
    "qualification": "You are an expert AI Presale Solution Architect. Conduct a comprehensive Lead Qualification assessment using the BANT framework (Budget, Authority, Need, Timeline/Engagement) and assign a qualification status."
}

def format_transcript(conv_data):
    if isinstance(conv_data, str):
        try:
            conv_data = json.loads(conv_data)
        except Exception:
            return conv_data
    turns = []
    for turn in conv_data:
        speaker = "Customer" if turn.get("speaker") == "customer" else "Sales Rep"
        turns.append(f"{speaker}: {turn.get('message', '').strip()}")
    return "\n".join(turns)

def parse_scenario(scenario_data):
    if isinstance(scenario_data, str):
        try:
            return json.loads(scenario_data)
        except Exception:
            return {}
    return scenario_data if isinstance(scenario_data, dict) else {}

def build_tasks(record):
    scenario = parse_scenario(record.get("scenario", {}))
    persona = scenario.get("customer_persona", {})
    transcript = format_transcript(record.get("conversation", []))
    company = record.get("company_name", persona.get("company", "Prospect Company"))
    product = record.get("product_name", "SaaS Solution")
    pain_points = persona.get("pain_points", [])
    needs = persona.get("needs", [])
    budget_sens = persona.get("budget_sensitivity", "medium")
    authority = persona.get("decision_making_authority", "evaluator")
    role = persona.get("role", "Prospect")
    outcome = record.get("outcome", 0)
    engagement = record.get("customer_engagement", 0.5)
    conv_id = record.get("conversation_id", "")

    # 1. Pain Points
    t1_p = f"Sales Conversation Transcript:\n```\n{transcript}\n```\n\nIdentify and list the customer's primary pain points and operational bottlenecks:"
    t1_r = "Based on the conversation, the customer's key pain points are:\n" + ("\n".join([f"- {p}" for p in pain_points]) if pain_points else "- Workflow inefficiencies and tooling complexity.")
    
    # 2. Requirements
    t2_p = f"Sales Conversation Transcript:\n```\n{transcript}\n```\n\nExtract the customer's functional and technical requirements for the solution:"
    t2_r = "The core customer requirements and solution needs identified:\n" + ("\n".join([f"- {n}" for n in needs]) if needs else "- An intuitive solution that simplifies team workflows.")
    
    # 3. Intent Classification
    if budget_sens == "high":
        intent, just = "Pricing & ROI Evaluation", "Customer actively inquires about pricing and ROI compared to existing tool costs."
    elif outcome == 1 and authority in ["decision_maker", "signer"]:
        intent, just = "Commercial Negotiation", "Customer has decision authority and is moving towards procurement terms."
    elif any("train" in str(p).lower() for p in pain_points):
        intent, just = "Implementation & Onboarding Assessment", "Customer focuses heavily on team onboarding curve and adoption support."
    else:
        intent, just = "Product Capability Evaluation", "Customer is evaluating product features against their internal workflows."
    t3_p = f"Sales Conversation Transcript:\n```\n{transcript}\n```\n\nClassify the customer's primary intent and provide a brief justification:"
    t3_r = f"**Primary Intent**: {intent}\n\n**Analysis**: {just}"

    # 4. Objections
    objs = []
    if budget_sens == "high":
        objs.append("Cost & ROI Concerns: Subscription pricing scrutiny and budget constraints.")
    if any("train" in str(p).lower() for p in pain_points):
        objs.append("Learning Curve: Team is busy and may face adoption hurdles.")
    if not objs:
        objs.append("Bandwidth & Implementation Timing: Concerns around rollout bandwidth.")
    t4_p = f"Sales Conversation Transcript:\n```\n{transcript}\n```\n\nIdentify and analyze the customer's objections and hesitations:"
    t4_r = "The customer raised the following key objections during the discussion:\n" + "\n".join([f"- {o}" for o in objs])

    # 5. Lead Qualification (BANT)
    status = "Qualified (High Potential - Won)" if outcome == 1 else "Unqualified / At-Risk (Nurture Required)"
    t5_p = f"Sales Conversation Transcript:\n```\n{transcript}\n```\n\nCompany: {company}\nProduct Discussed: {product}\nConduct a BANT Lead Qualification analysis and deliver the evaluation:"
    t5_r = (
        f"### BANT Lead Qualification Report\n"
        f"- **Budget**: Sensitivity is {budget_sens.upper()}. {'Clear budget approval path.' if outcome == 1 else 'Requires concrete ROI proof to unlock spend.'}\n"
        f"- **Authority**: Stakeholder role is {role} ({authority.capitalize()}). Needs team buy-in.\n"
        f"- **Need**: Critical urgency around workflow friction and operational efficiency.\n"
        f"- **Timeline / Engagement**: Customer engagement score {engagement:.1f}/1.0.\n\n"
        f"**Lead Qualification Verdict**: {status}\n"
        f"**Next Presale Step**: {'Deliver technical architecture demo & contract proposal.' if outcome == 1 else 'Send case studies addressing ROI and schedule follow-up discovery session.'}"
    )

    return [
        {"task": "pain_point", "messages": [{"role": "system", "content": SYSTEM_PROMPTS["pain_point"]}, {"role": "user", "content": t1_p}, {"role": "assistant", "content": t1_r}]},
        {"task": "requirement", "messages": [{"role": "system", "content": SYSTEM_PROMPTS["requirement"]}, {"role": "user", "content": t2_p}, {"role": "assistant", "content": t2_r}]},
        {"task": "intent", "messages": [{"role": "system", "content": SYSTEM_PROMPTS["intent"]}, {"role": "user", "content": t3_p}, {"role": "assistant", "content": t3_r}]},
        {"task": "objection", "messages": [{"role": "system", "content": SYSTEM_PROMPTS["objection"]}, {"role": "user", "content": t4_p}, {"role": "assistant", "content": t4_r}]},
        {"task": "qualification", "messages": [{"role": "system", "content": SYSTEM_PROMPTS["qualification"]}, {"role": "user", "content": t5_p}, {"role": "assistant", "content": t5_r}]}
    ]

# Load streaming dataset to avoid loading full 100k conversations
print("Loading DeepMostInnovations/saas-sales-conversations...")
raw_ds = load_dataset("DeepMostInnovations/saas-sales-conversations", split="train", streaming=True)

TARGET_CONVERSATIONS = 2000
TRAIN_CONVS = 1600
VAL_CONVS = 400

conversations = []
for idx, item in enumerate(raw_ds):
    record = {k: v for k, v in item.items() if not k.startswith("embedding_")}
    conversations.append(record)
    if len(conversations) >= TARGET_CONVERSATIONS:
        break
    if (idx + 1) % 500 == 0:
        print(f"Collected {len(conversations)}/{TARGET_CONVERSATIONS} conversations...")

print(f"Total collected conversations: {len(conversations)}")
random.seed(42)
random.shuffle(conversations)

train_conv_subset = conversations[:TRAIN_CONVS]
val_conv_subset = conversations[TRAIN_CONVS:TRAIN_CONVS + VAL_CONVS]

train_data = []
for c in train_conv_subset:
    train_data.extend(build_tasks(c))

val_data = []
for c in val_conv_subset:
    val_data.extend(build_tasks(c))

print(f"\nDataset prepared successfully!")
print(f"Train samples: {len(train_data)} (from {len(train_conv_subset)} conversations)")
print(f"Val samples:   {len(val_data)} (from {len(val_conv_subset)} conversations)")


In [ ]:
# Step 3: Load Qwen-4B Model and Tokenizer with 4-bit NF4 Quantization
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training

MODEL_ID = "Qwen/Qwen1.5-4B-Chat"

print(f"Loading tokenizer for {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    padding_side="right"
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 4-bit NormalFloat (NF4) Quantization Config with float16 compute dtype for Tesla T4
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print(f"Loading base model {MODEL_ID} in 4-bit...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
print("Base model prepared for 4-bit training!")


In [ ]:
# Step 4: Configure PEFT / LoRA Adapter Specification
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Format chat conversations using Qwen Chat Template
def format_prompts(batch):
    formatted_texts = []
    for messages in batch["messages"]:
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
        formatted_texts.append(text)
    return {"text": formatted_texts}

train_dataset = Dataset.from_list(train_data).map(format_prompts, batched=True)
val_dataset = Dataset.from_list(val_data).map(format_prompts, batched=True)

print("Sample formatted training prompt:")
print("-" * 50)
print(train_dataset[0]["text"][:600] + "...")
print("-" * 50)


In [ ]:
# Step 5: Setup Training Arguments & SFTTrainer
# In 4-bit QLoRA, the 4-bit weights are dequantized to float16 during forward pass.
# Keeping fp16=False disables PyTorch's GradScaler, eliminating mixed-precision unscale crashes on Tesla T4.
from trl import SFTTrainer, SFTConfig

OUTPUT_DIR = "/kaggle/working/qwen_presale_qlora"

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    dataset_text_field="text",
    max_length=1536,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,  # Effective batch size = 16
    optim="paged_adamw_8bit",
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=30,
    fp16=False,
    bf16=False,
    gradient_checkpointing=True,
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=250,
    save_strategy="steps",
    save_steps=250,
    save_total_limit=2,
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    args=training_args
)

print("Starting QLoRA Fine-Tuning...")
trainer.train()
print("Training Complete!")


In [ ]:
# Step 6: Save LoRA Adapter & Tokenizer
FINAL_ADAPTER_DIR = "/kaggle/working/qwen_presale_adapter"

print(f"Saving LoRA adapter to {FINAL_ADAPTER_DIR}...")
trainer.model.save_pretrained(FINAL_ADAPTER_DIR)
tokenizer.save_pretrained(FINAL_ADAPTER_DIR)

print("Adapter files saved:")
!ls -lh {FINAL_ADAPTER_DIR}


In [ ]:
# Step 7: Test Inference on a New Sales Conversation across all 5 Presale Tasks
test_transcript = """Customer: Hi there, our engineering team is spending over 20 hours a week maintaining legacy CI/CD scripts and our test suites keep failing silently.
Sales Rep: That sounds frustrating. Are you looking to replace your current pipelines or augment them with automated orchestration?
Customer: Definitely augment first. But our main concern is security compliance (SOC2) and whether our junior engineers can pick this up quickly without a 2-month training period. Also, what is your pricing tier for 50 seats?"""

tasks_to_test = [
    ("Pain Point Extraction", SYSTEM_PROMPTS["pain_point"], "Identify and list the customer's primary pain points:"),
    ("Requirement Extraction", SYSTEM_PROMPTS["requirement"], "Extract the customer's functional and technical requirements:"),
    ("Intent Classification", SYSTEM_PROMPTS["intent"], "Classify the customer's primary intent and provide a brief justification:"),
    ("Objection Extraction", SYSTEM_PROMPTS["objection"], "Identify and analyze the customer's objections and hesitations:"),
    ("Lead Qualification", SYSTEM_PROMPTS["qualification"], "Company: FinTech SaaS\nProduct Discussed: CloudOrchestrator Pro\nConduct a BANT Lead Qualification analysis:")
]

infer_model = trainer.model
infer_model.eval()

print("=== TESTING FINE-TUNED PRE-SALE LLM ===\n")
for task_name, sys_prompt, user_query in tasks_to_test:
    print(f"\n==================== [TASK: {task_name.upper()}] ====================")
    prompt_messages = [
        {"role": "system", "content": sys_prompt},
        {"role": "user", "content": f"Sales Conversation Transcript:\n```\n{test_transcript}\n```\n\n{user_query}"}
    ]
    
    input_text = tokenizer.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt").to(infer_model.device)
    
    with torch.no_grad():
        outputs = infer_model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.2,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id
        )
    
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    print(response.strip())
